# AI-Powered EdTech & Learning Analytics: Africa and USA

This notebook explores World Bank education and digital-access indicators for **15 African countries and the USA** (2015-2025) to understand where AI-powered EdTech can realistically reach learners today.

| Indicator | World Bank code | Why it matters for EdTech |
|---|---|---|
| Primary completion rate | `SE.PRM.CMPT.ZS` | Foundational learning pipeline |
| Government education expenditure (% GDP) | `SE.XPD.TOTL.GD.ZS` | Public capacity to fund digital learning |
| Tertiary enrollment (% gross) | `SE.TER.ENRR` | Demand for advanced / online learning |
| Internet users (% population) | `IT.NET.USER.ZS` | Physical access to digital learning tools |
| Youth literacy (% ages 15-24) | `SE.ADT.1524.LT.ZS` | Ability to use text-based learning platforms |

Data are read directly from the raw CSVs in `data/raw/` (columns: `country_id, country_name, indicator_id, indicator_name, year, value`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

# Works whether the notebook is started from the project root or notebooks/
RAW_DIR = Path("data/raw") if Path("data/raw").exists() else Path("../data/raw")
FILES = {
    "primary_completion_rate": "wb_primary_completion_rate.csv",
    "education_expenditure_pct_gdp": "wb_education_expenditure.csv",
    "tertiary_enrollment_rate": "wb_tertiary_enrollment.csv",
    "internet_users_pct": "wb_internet_users.csv",
    "youth_literacy_rate": "wb_youth_literacy.csv",
}
LABELS = {
    "primary_completion_rate": "Primary completion %",
    "education_expenditure_pct_gdp": "Edu spend % GDP",
    "tertiary_enrollment_rate": "Tertiary enrollment %",
    "internet_users_pct": "Internet users %",
    "youth_literacy_rate": "Youth literacy %",
}
print("Reading raw data from", RAW_DIR.resolve())

## 1. Data loading and cleaning

In [ ]:
frames = []
for key, fname in FILES.items():
    df = pd.read_csv(RAW_DIR / fname)
    df["indicator_key"] = key
    frames.append(df)
    print(f"{fname:<34} rows={len(df):>4}  years={df.year.min()}-{df.year.max()}  countries={df.country_id.nunique()}")

raw = pd.concat(frames, ignore_index=True)
raw["year"] = pd.to_numeric(raw["year"], errors="coerce").astype("Int64")
raw["value"] = pd.to_numeric(raw["value"], errors="coerce")
raw = (raw.dropna(subset=["year", "value"])
          .drop_duplicates(subset=["country_id", "indicator_id", "year"], keep="last"))
raw["year"] = raw["year"].astype(int)
raw["country_name"] = raw["country_name"].replace({"Egypt, Arab Rep.": "Egypt"})
raw["region"] = np.where(raw["country_id"] == "US", "USA", "Africa")
print("Clean observations:", len(raw))
raw.head()

## 2. Exploratory data analysis

In [ ]:
# Descriptive statistics per indicator and region
stats = (raw.groupby(["indicator_key", "region"])["value"]
            .describe()[["count", "mean", "std", "min", "50%", "max"]]
            .round(2))
stats

In [ ]:
# Data coverage: number of reported years per country and indicator
coverage = raw.pivot_table(index="country_name", columns="indicator_key",
                           values="value", aggfunc="count", fill_value=0)
coverage.rename(columns=LABELS)

In [ ]:
# Latest available value for every country / indicator
latest = (raw.sort_values("year")
             .groupby(["country_id", "country_name", "region", "indicator_key"])
             .tail(1))
latest_wide = latest.pivot_table(index=["country_id", "country_name", "region"],
                                 columns="indicator_key", values="value").reset_index()
latest_wide.columns.name = None

# EdTech readiness score: weighted mean of available components
weights = {"internet_users_pct": 0.4, "youth_literacy_rate": 0.3, "tertiary_enrollment_rate": 0.3}
comp = latest_wide[list(weights)].clip(upper=100)
w = pd.Series(weights)
latest_wide["edtech_readiness_score"] = ((comp.fillna(0) * w).sum(axis=1)
                                         / (comp.notna() * w).sum(axis=1)).round(2)
latest_wide["components"] = comp.notna().sum(axis=1)
latest_wide = latest_wide.sort_values("edtech_readiness_score", ascending=False)
latest_wide.round(2)

## 3. Visualizations

### 3.1 Primary completion rate by country (latest year)

In [ ]:
pcr = latest[latest.indicator_key == "primary_completion_rate"].sort_values("value")
fig = px.bar(pcr, x="value", y="country_name", color="region", orientation="h",
             text=pcr["value"].round(1), hover_data=["year"],
             color_discrete_map={"Africa": "#E07A1F", "USA": "#1F5AA6"},
             labels={"value": "Primary completion rate (% of relevant age group)", "country_name": ""},
             title="Primary completion rate - latest reported year")
fig.add_vline(x=100, line_dash="dot", line_color="grey")
fig.update_layout(height=550)
fig.show()

### 3.2 Internet users trend - top African countries vs USA

In [ ]:
inet = raw[raw.indicator_key == "internet_users_pct"]
top_africa = (latest[(latest.indicator_key == "internet_users_pct") & (latest.region == "Africa")]
              .nlargest(6, "value")["country_id"].tolist())
trend = inet[inet.country_id.isin(top_africa + ["US"])].sort_values("year")
fig = px.line(trend, x="year", y="value", color="country_name", markers=True,
              labels={"value": "Internet users (% of population)", "year": "Year", "country_name": "Country"},
              title="Internet adoption 2015-2024: top 6 African countries vs USA")
fig.update_traces(selector=dict(name="United States"), line=dict(width=4, dash="dash"))
fig.show()

growth = inet.sort_values("year").groupby("country_name")["value"].agg(["first", "last"])
growth["gain_pp"] = (growth["last"] - growth["first"]).round(1)
growth.sort_values("gain_pp", ascending=False).head(8)

### 3.3 Internet access vs tertiary enrollment

In [ ]:
pairs = (raw[raw.indicator_key.isin(["internet_users_pct", "tertiary_enrollment_rate"])]
         .pivot_table(index=["country_name", "region", "year"], columns="indicator_key", values="value")
         .dropna().reset_index())
r = pairs["internet_users_pct"].corr(pairs["tertiary_enrollment_rate"])
fig = px.scatter(pairs, x="internet_users_pct", y="tertiary_enrollment_rate", color="region",
                 hover_name="country_name", hover_data=["year"],
                 color_discrete_map={"Africa": "#E07A1F", "USA": "#1F5AA6"},
                 labels={"internet_users_pct": "Internet users (% of population)",
                         "tertiary_enrollment_rate": "Tertiary enrollment (% gross)"},
                 title=f"Internet access vs tertiary enrollment (country-years, Pearson r = {r:.2f})")
# Simple least-squares trend line computed with numpy
slope, intercept = np.polyfit(pairs["internet_users_pct"], pairs["tertiary_enrollment_rate"], 1)
xs = np.linspace(pairs["internet_users_pct"].min(), pairs["internet_users_pct"].max(), 50)
fig.add_trace(go.Scatter(x=xs, y=slope * xs + intercept, mode="lines",
                         name="OLS trend", line=dict(color="grey", dash="dot")))
fig.show()
print(f"n = {len(pairs)} country-years, r = {r:.3f}, slope = {slope:.2f} enrollment pp per internet pp")

### 3.4 Heatmap: all indicators by country (min-max normalised)

In [ ]:
ind_cols = list(FILES)
heat = latest_wide.set_index("country_name")[ind_cols]
norm = (heat - heat.min()) / (heat.max() - heat.min())
fig = px.imshow(norm.rename(columns=LABELS).round(2), text_auto=True, aspect="auto",
                color_continuous_scale="YlGnBu", zmin=0, zmax=1,
                title="Latest indicator values, normalised 0-1 across countries (blank = not reported)")
fig.update_layout(height=600, xaxis_title="", yaxis_title="")
fig.show()

### 3.5 Government education expenditure (% of GDP)

In [ ]:
spend = latest[latest.indicator_key == "education_expenditure_pct_gdp"].sort_values("value", ascending=False)
africa_avg = spend[spend.region == "Africa"]["value"].mean()
fig = px.bar(spend, x="country_name", y="value", color="region", text=spend["value"].round(1),
             hover_data=["year"],
             color_discrete_map={"Africa": "#E07A1F", "USA": "#1F5AA6"},
             labels={"value": "Education expenditure (% of GDP)", "country_name": ""},
             title="Government expenditure on education - latest reported year")
fig.add_hline(y=4, line_dash="dash", line_color="green",
              annotation_text="UNESCO Education 2030 lower benchmark (4%)")
fig.add_hline(y=africa_avg, line_dash="dot", line_color="#E07A1F",
              annotation_text=f"Africa average {africa_avg:.1f}%", annotation_position="bottom right")
fig.show()

### 3.6 Radar chart: EdTech readiness components by country

In [ ]:
radar_countries = latest_wide.head(6)["country_name"].tolist()
if "United States" not in radar_countries:
    radar_countries.append("United States")
axes = ["internet_users_pct", "youth_literacy_rate", "tertiary_enrollment_rate",
        "primary_completion_rate", "edtech_readiness_score"]
axis_labels = [LABELS.get(a, "EdTech readiness score") for a in axes]

fig = go.Figure()
for name in radar_countries:
    row = latest_wide.set_index("country_name").loc[name, axes].astype(float).clip(upper=100).fillna(0)
    values = row.tolist()
    fig.add_trace(go.Scatterpolar(r=values + values[:1], theta=axis_labels + axis_labels[:1],
                                  fill="toself", name=name, opacity=0.55))
fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
                  title="EdTech readiness profile - top 6 African countries vs USA (missing = 0)",
                  height=600)
fig.show()

px.bar(latest_wide.sort_values("edtech_readiness_score"), x="edtech_readiness_score", y="country_name",
       color="region", orientation="h", text="edtech_readiness_score", hover_data=["components"],
       color_discrete_map={"Africa": "#E07A1F", "USA": "#1F5AA6"},
       labels={"edtech_readiness_score": "EdTech readiness score (0-100)", "country_name": ""},
       title="EdTech readiness score ranking").show()

## 4. Africa vs USA: what the data says

The comparison below averages each country's latest value per indicator. The USA is a single country while Africa is the mean of 15 very different education systems, so the African spread (min / max) matters as much as the average.

In [ ]:
region_cmp = latest.groupby(["indicator_key", "region"])["value"].mean().unstack().round(2)
spread = latest[latest.region == "Africa"].groupby("indicator_key")["value"].agg(["min", "max"]).round(2)
region_cmp = region_cmp.join(spread.add_prefix("africa_"))
region_cmp["usa_minus_africa"] = (region_cmp["USA"] - region_cmp["Africa"]).round(2)
region_cmp.rename(index=LABELS)

**Reading the comparison**

* **Digital access is the widest gap.** US internet use is around 95% of the population; the African average of the sample is roughly half that, with Morocco (91%) and South Africa (78%) close to US levels while Uganda and Zambia remain below 20%.
* **Access is improving fast.** Ghana (+49 pp), Egypt (+37 pp), Senegal (+38 pp) and Morocco (+34 pp) recorded the largest internet gains since 2015 - the addressable market for mobile-first AI tutors is growing every year.
* **Tertiary enrollment is the steepest cliff.** The US gross tertiary enrollment rate (~79%) is several times the African average (~19%); internet access and tertiary enrollment are strongly correlated across country-years.
* **Funding is not the binding constraint everywhere.** Several African countries (Senegal, South Africa, Morocco) spend a larger share of GDP on education than the US, yet outcomes differ - efficiency and delivery models matter, which is exactly where data-driven learning analytics can help.
* **Youth literacy is relatively high** (most countries 80-97%), so the foundational skill needed for text-based digital learning is largely present; the US does not report this indicator to the World Bank.

## 5. Conclusion - key findings

1. **The EdTech readiness score is led by the USA (~88)**, followed by Morocco, Egypt, South Africa and Ghana (63-73). Countries with incomplete data (e.g. Kenya, which lacks a recent youth-literacy figure) are scored only on the components they report, so their rank should be read with care.
2. **The digital divide is closing but still large:** internet use in the sample's African countries roughly doubled between 2015 and 2024, yet the median is still near 41%.
3. **Internet access is a strong predictor of tertiary participation** (Pearson r ~ 0.8 across country-years), supporting investment in connectivity as a lever for higher-education access and online learning.
4. **Primary completion remains uneven** - above 95% in Egypt, Ghana, South Africa, Kenya and Morocco but near 53-55% in Uganda and Ethiopia, where offline-capable, low-bandwidth AI learning tools are most relevant.
5. **Spending alone does not guarantee outcomes**; learning analytics that track completion, progression and engagement can help ministries target resources where they move outcomes most.

**Next steps:** enrich the pipeline with learning-outcome data (e.g. learning poverty, assessment scores), mobile broadband cost, and electricity access to refine the readiness score.